# Qwen2.5 3B CHLT Reasoner Final Adapter

Purpose: train the model to answer conceptual CHLT physics questions with evidence-grounded JSON. This adapter is separate from numeric calculation.


In [ ]:
!pip -q install -U "transformers>=4.46" "datasets>=2.20" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.46.1" safetensors


In [ ]:
import json, os, time
from pathlib import Path

ADAPTER_KEY = 'chlt_reasoner_final'
ADAPTER_NAME = 'qwen25_3b_chlt_reasoner_final_adapter'
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR = Path('/kaggle/working') / ADAPTER_NAME
MAX_SEQ_LENGTH = 1536
EPOCHS = 3
LR = 1.5e-4
REQUIRED_KEYS = {'question_kind', 'topic', 'concept', 'answer_type', 'answer', 'evidence', 'confidence'}
FORBIDDEN_KEYS = {'python_code'}
print('Output:', OUTPUT_DIR)


In [ ]:
def find_jsonl(split):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    wanted = {f'{split}.seed.jsonl', f'{ADAPTER_KEY}_{split}.jsonl', f'physics_{ADAPTER_KEY}_{split}.jsonl'}
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob('*.jsonl'):
            low = str(path).lower()
            if path.name in wanted and ADAPTER_KEY in low:
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'Could not find {split} jsonl for {ADAPTER_KEY}. Upload the adapter folder as a Kaggle dataset.')
    def priority(path):
        name = path.name
        if name == f'{ADAPTER_KEY}_{split}.jsonl':
            return (0, len(str(path)))
        if name == f'physics_{ADAPTER_KEY}_{split}.jsonl':
            return (1, len(str(path)))
        if name == f'{split}.seed.jsonl':
            return (9, len(str(path)))
        return (5, len(str(path)))
    return sorted(candidates, key=priority)[0]

TRAIN_FILE = find_jsonl('train')
VALID_FILE = find_jsonl('valid')
print('Train:', TRAIN_FILE)
print('Valid:', VALID_FILE)


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def validate_rows(rows, split):
    bad = []
    for row in rows:
        messages = row.get('messages') or []
        if len(messages) < 3:
            bad.append((row.get('id'), 'messages_too_short'))
            continue
        try:
            obj = json.loads(messages[-1]['content'])
        except Exception as exc:
            bad.append((row.get('id'), f'invalid_json:{exc}'))
            continue
        missing = sorted(REQUIRED_KEYS - set(obj))
        forbidden = sorted(FORBIDDEN_KEYS & set(obj))
        if missing or forbidden:
            bad.append((row.get('id'), f'missing={missing}; forbidden={forbidden}'))
    if bad:
        print(split, 'bad rows:', bad[:10])
        raise ValueError(f'{split} validation failed: {len(bad)} bad rows')
    print(split, 'validated rows:', len(rows))

train_rows = load_jsonl(TRAIN_FILE)
valid_rows = load_jsonl(VALID_FILE)
validate_rows(train_rows, 'train')
validate_rows(valid_rows, 'valid')


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def render_text(row):
    return tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)

train_texts = [render_text(row) for row in train_rows]
valid_texts = [render_text(row) for row in valid_rows]
train_ds = Dataset.from_dict({'text': train_texts})
valid_ds = Dataset.from_dict({'text': valid_texts})
print('Rendered train rows:', len(train_ds))
print('Rendered valid rows:', len(valid_ds))
print(train_ds[0]['text'][:1000])

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(tokenize, batched=True, remove_columns=valid_ds.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
common_args = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)
try:
    args = TrainingArguments(eval_strategy='steps', **common_args)
except TypeError:
    args = TrainingArguments(evaluation_strategy='steps', **common_args)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=valid_tok, data_collator=collator)
trainer.train()

adapter_dir = OUTPUT_DIR / 'adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
metadata = {
    'base_model': BASE_MODEL,
    'adapter_purpose': ADAPTER_KEY,
    'train_rows': len(train_rows),
    'valid_rows': len(valid_rows),
    'max_seq_length': MAX_SEQ_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'required_output_keys': sorted(REQUIRED_KEYS),
    'forbidden_output_keys': sorted(FORBIDDEN_KEYS),
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
(adapter_dir / 'training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved adapter to', adapter_dir)
print(json.dumps(metadata, indent=2))


## Quick Generation Validation

Checks that the trained adapter returns valid conceptual JSON.


In [ ]:
import json, torch, pandas as pd

model.eval()

def extract_json_object(text):
    first = text.find('{')
    last = text.rfind('}')
    if first < 0 or last <= first:
        return None
    try:
        return json.loads(text[first:last+1])
    except Exception:
        return None

def validate_chlt_obj(obj):
    if not isinstance(obj, dict):
        return {'json_ok': False, 'schema_ok': False, 'answer_ok': False, 'forbidden_ok': False}
    schema_ok = REQUIRED_KEYS.issubset(set(obj)) and isinstance(obj.get('evidence'), list) and isinstance(obj.get('confidence'), (int, float))
    forbidden_ok = not bool(FORBIDDEN_KEYS & set(obj))
    answer_type = obj.get('answer_type')
    answer = obj.get('answer')
    answer_ok = bool(answer) and (answer_type != 'yes_no' or answer in {'Yes', 'No', 'Uncertain'})
    return {'json_ok': True, 'schema_ok': schema_ok, 'answer_ok': answer_ok, 'forbidden_ok': forbidden_ok}

def generate_chlt(question, max_new_tokens=300):
    messages = [
        {'role': 'system', 'content': 'You answer conceptual physics questions. Return exactly one JSON object. If options are present, the answer must match one option. If uncertain, answer Uncertain. Do not output code.'},
        {'role': 'user', 'content': 'Question: ' + question.strip()},
    ]
    encoded = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True)
    if hasattr(encoded, 'to'):
        encoded = encoded.to(model.device)
    else:
        encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        output_ids = model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    prompt_len = encoded['input_ids'].shape[-1]
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()

checks = [
    'A series RLC circuit has R = 20 ohm, L = 0.5 H, and C = 20 uF. If the source frequency is 50.33 Hz, does resonance occur?',
    'In an ideal LC circuit with no resistance, is total electromagnetic energy conserved?',
    'If a conductor loop has no change in magnetic flux, is an induced emf produced?',
    'At resonance in a series RLC circuit, is the impedance equal to the resistance?',
]
rows = []
for idx, question in enumerate(checks, 1):
    raw = generate_chlt(question)
    obj = extract_json_object(raw)
    metrics = validate_chlt_obj(obj)
    rows.append({'idx': idx, 'question': question, **metrics, 'raw': raw[:1000]})
    print(idx, metrics)
    print(raw[:900])
    print('-' * 80)

out_csv = '/kaggle/working/chlt_reasoner_quick_generation.csv'
pd.DataFrame(rows).to_csv(out_csv, index=False)
print('Saved:', out_csv)
